In [ ]:
import json
import os
import time

import bibtexparser
import flatdict as fd
import numpy as np
import pandas as pd

from pynxtools_apm.examples.oasisb.oasisb_bibliography import (
    get_bibliographical_metadata,
)
from pynxtools_apm.examples.oasisb.oasisb_openalex import get_data_for_doi_from_openalex
from pynxtools_apm.examples.oasisb.oasisb_utils import get_project_id

rng = np.random.default_rng(seed=42)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory: str = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype="str",
).fillna("")
with open(f"{src_directory}{os.sep}aaa_legacy_data.bib") as fp:
    bib = bibtexparser.load(fp).entries_dict

project_range: tuple[int, int] = (1, 156)

## Query for each project bibliographical metadata from OpenAlex

Querying metadata in addition to the DOI allows to cross-check authors, institutions, and copyright relevant details.

In [ ]:
api_queries_cnt: int = 0
api_queries_max: int = 10
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if project_range[0] <= int(row.id) <= project_range[1]:
            project_id: str = get_project_id(f"{row.id}")
            data_and_paper: list[str] = get_bibliographical_metadata(
                bib, row.project_name
            )

            n_queries: int = get_data_for_doi_from_openalex(bib, data_and_paper)

            api_queries_cnt += n_queries  # sleep only when necessary
            if api_queries_cnt >= api_queries_max:
                sleep: float = float(rng.uniform(1, 10))
                print(f"Sleeping for {sleep}s")
                time.sleep(sleep)
                api_queries_cnt = 0
print("Batch querying queue done")

***

## Normalize author field cross-checking against OpenAlex

Often author names are abbreviated, e.g. Breen, A. J. instead of Breen, Andrew John.
Here we remove abbreviations.

In [ ]:
with open(f"{src_directory}{os.sep}aaa_legacy_data.bib") as fp:
    bib = bibtexparser.load(fp).entries_dict
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if row.legal == "1" and row.use == "1":
            data_and_paper: list[str] = get_bibliographical_metadata(
                bib,
                row.project_name,
                verbose=False,
            )
            print(row.project_name)
            print(bib[data_and_paper[0]]["author"])
            continue
            openalex_authors: list[str] = []
            openalex_file = (
                f"{os.getcwd()}{os.sep}openalex{os.sep}{data_and_paper[0]}.json"
            )
            try:
                if os.path.isfile(openalex_file):
                    with open(openalex_file, encoding="utf-8") as fp:
                        openalex = fd.FlatDict(json.load(fp), "/")
                        if "authorships" in openalex:
                            for dct in openalex["authorships"]:
                                flat = fd.FlatDict(dct, "/")
                                if "raw_author_name" in flat:
                                    openalex_authors.append(
                                        flat["raw_author_name"].strip()
                                    )
            except TypeError:
                pass
            bibtex_authors: list[str] = []
            for bibtex_author in [
                value.strip() for value in bib[data_and_paper[0]]["author"].split("and")
            ]:
                bibtex_authors.append(bibtex_author)
            if openalex_authors != bibtex_authors:
                print(f"{row.project_name}")
                print(openalex_authors)
                print(bibtex_authors)
            del openalex_authors, bibtex_authors